[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C69_Agent_Security_Course/03_sandbox_permissions/03_sandbox_permissions.ipynb)

# 03 · 沙箱与权限边界（能力四维 / 隔离层级 / 确认疲劳 / TOCTOU / 审计 / 变更影响面）

目标：把「最小权限」从口号变成**一个能算出来的集合**，并把确认与审计做成可验证的属性。

本 notebook 你会亲手实现：
1. **能力四维模型** —— 操作 / 范围 / 信任门槛 / 可逆性，四个条件合取
2. **路径范围检查的绕过与修法** —— `startswith` 为什么不够
3. **最小权限求解** —— 从任务步骤反推权限集，并与「按角色授予」对比
4. **确认疲劳的模拟** —— 确认频率如何摧毁确认的有效性
5. **TOCTOU** —— 确认的内容与执行的内容不一致，以及冻结动作对象的修法
6. **审计与 provenance 链** —— 唯一无法事后补上的字段
7. **变更影响面与季度锚点** —— 让「每次只加一点」的累积变得可见

> 心智模型：**沙箱隔离的是「代码执行的副作用」，
> 而 agent 的主要能力来自它被显式授予的 API 调用——那些调用是「合法」的，沙箱不会拦。
> 所以权限边界通常比沙箱更重要。**

## 0 · 环境与能力四维模型

In [ ]:
import os, json, math, re, hashlib, itertools, posixpath
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import numpy as np

L0, L1, L2, L3 = 0, 1, 2, 3
LEVEL_NAME = {0: 'L0 不受信', 1: 'L1 半可信', 2: 'L2 用户', 3: 'L3 系统'}

def sha(obj, n=10):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:n]

@dataclass(frozen=True)
class Cap:
    """能力的四个维度。frozen=True 让它不可变、可哈希。"""
    op: str                       # ① 操作
    scope: tuple                  # ② 范围（允许的目标模式）
    min_trust: int                # ③ 信任门槛
    reversible: bool              # ④ 可逆性
    egress: bool = False          # 是否构成对外通信（04 模块）

CAPS = {
    'read_workspace': Cap('read',  ('/work/**',),                L0, True),
    'read_home':      Cap('read',  ('/home/**',),                L2, True),
    'write_workspace':Cap('write', ('/work/**',),                L1, False),
    'send_email':     Cap('send',  ('*@corp.test',),             L2, False, True),
    'http_get':       Cap('fetch', ('*.corp.test', '*.pypi.org'), L1, True, True),
}
for name, c in CAPS.items():
    print(f'{name:<18} op={c.op:<6} scope={c.scope} '
          f'min_trust={LEVEL_NAME[c.min_trust]:<10} 可逆={c.reversible} 外通={c.egress}')
print('\n✅ 四个维度都是显式的——而多数系统只做了第一维（「有没有这个工具」）。')

## 1 · 范围检查：`startswith` 为什么不够

In [ ]:
def scope_match_naive(target, patterns):
    """❌ 朴素前缀匹配。两个错误：不规范化路径、比较时去掉了尾部分隔符。
    后者是很常见的写法（「把通配符和斜杠都去掉再比」）。"""
    for p in patterns:
        prefix = p.replace('**', '').rstrip('/')      # ← '/work/**' → '/work'
        if target.startswith(prefix):
            return True
    return False

def scope_match_safe(target, patterns):
    """✓ 先规范化（解析 .. 与多余分隔符）成绝对路径，再比较。"""
    norm = posixpath.normpath(posixpath.join('/', target))
    for p in patterns:
        base = posixpath.normpath(posixpath.join('/', p.replace('**', '')))
        # 必须是 base 本身或 base 下的子路径
        if norm == base or norm.startswith(base.rstrip('/') + '/'):
            return True
    return False

WORKSPACE = ('/work/**',)
CASES = [
    '/work/notes.txt',
    '/work/sub/dir/a.py',
    '/work/../home/user/.ssh/id_rsa',        # ← 经典绕过
    '/work/./../../etc/shadow',
    '/workspace_other/secret.txt',           # ← 前缀相同但不是子目录
    '/home/user/.ssh/id_rsa',
]
print(f"{'目标路径':<40}{'朴素':>8}{'安全':>8}")
for t in CASES:
    print(f'{t:<40}{str(scope_match_naive(t, WORKSPACE)):>8}'
          f'{str(scope_match_safe(t, WORKSPACE)):>8}')

assert scope_match_naive('/work/../home/user/.ssh/id_rsa', WORKSPACE) is True
assert scope_match_safe('/work/../home/user/.ssh/id_rsa', WORKSPACE) is False
assert scope_match_naive('/workspace_other/secret.txt', WORKSPACE) is True
assert scope_match_safe('/workspace_other/secret.txt', WORKSPACE) is False
assert scope_match_safe('/work/sub/dir/a.py', WORKSPACE) is True
assert scope_match_safe('/home/user/.ssh/id_rsa', WORKSPACE) is False
print('\n✅ 两类绕过都被修掉了：')
print('   ① `..` 穿越 —— 必须先 normpath 再比较')
print('   ② 前缀相同但不是子目录（/workspace_other） —— 必须在 base 后加分隔符')
print('   → 「读文件」收窄成「读 /work/」之后，「读 ~/.ssh/」这条路径变成**不可达**。')

In [ ]:
def glob_match(target, pattern):
    """域名/邮箱类的范围匹配。* 只匹配一段（不跨 .）—— 避免 evil.test 匹配 *.corp.test。"""
    rx = '^' + re.escape(pattern).replace(r'\*', r'[^.@]+') + '$'
    return re.match(rx, target) is not None

EMAIL_SCOPE = ('*@corp.test',)
HOST_SCOPE = ('*.corp.test', '*.pypi.org')
print(f"{'目标':<34}{'在范围内':>10}")
for t, scope in [('boss@corp.test', EMAIL_SCOPE), ('attacker@evil.test', EMAIL_SCOPE),
                 ('boss@corp.test.evil.test', EMAIL_SCOPE),
                 ('api.corp.test', HOST_SCOPE), ('evil.test', HOST_SCOPE),
                 ('api.corp.test.evil.test', HOST_SCOPE)]:
    print(f'{t:<34}{str(any(glob_match(t, p) for p in scope)):>10}')

assert glob_match('boss@corp.test', '*@corp.test') is True
assert glob_match('attacker@evil.test', '*@corp.test') is False
assert glob_match('boss@corp.test.evil.test', '*@corp.test') is False
assert glob_match('api.corp.test', '*.corp.test') is True
assert glob_match('api.corp.test.evil.test', '*.corp.test') is False
print('\n✅ 注意第三行与最后一行：**后缀伪装**（corp.test.evil.test）被正确拒绝——')
print('   这依赖于 `*` 不跨 `.` 这个约束。用 `.*` 的实现会被它绕过。')

## 2 · 四个条件的合取：只要一条不满足，动作就不可达

In [ ]:
@dataclass(frozen=True)
class Action:
    op: str
    target: str
    payload_sha: str
    ctx_trust: int
    provenance: tuple = ()

    def digest(self):
        return sha({'op': self.op, 'target': self.target,
                    'payload_sha': self.payload_sha, 'ctx_trust': self.ctx_trust})

def authorize(action, granted, confirmed=False):
    """四个条件合取。返回 (是否放行, 每一条的结果)。"""
    checks = {}
    cap = None
    for name in granted:
        c = CAPS[name]
        if c.op == action.op:
            cap = c
            break
    checks['granted'] = cap is not None
    if cap is None:
        return False, checks
    is_path = action.target.startswith('/')
    checks['scope'] = (scope_match_safe(action.target, cap.scope) if is_path
                       else any(glob_match(action.target, p) for p in cap.scope))
    checks['trust'] = action.ctx_trust >= cap.min_trust
    checks['reversible_or_confirmed'] = cap.reversible or confirmed
    return all(checks.values()), checks

GRANTED = ['read_workspace', 'write_workspace', 'send_email']
SCENARIOS = [
    ('读工作目录（L0 上下文）',    Action('read', '/work/a.txt', sha('x'), L0), False),
    ('读 home（L0 上下文）',       Action('read', '/home/u/.ssh/id_rsa', sha('x'), L0), False),
    ('写工作目录（L0，未确认）',   Action('write', '/work/b.txt', sha('x'), L0), False),
    ('写工作目录（L2，已确认）',   Action('write', '/work/b.txt', sha('x'), L2), True),
    ('发邮件给内部（L2，已确认）', Action('send', 'boss@corp.test', sha('x'), L2), True),
    ('发邮件给外部（L2，已确认）', Action('send', 'attacker@evil.test', sha('x'), L2), True),
    ('发邮件给内部（L0，已确认）', Action('send', 'boss@corp.test', sha('x'), L0), True),
]
print(f"{'场景':<30}{'放行':>6}  失败的检查")
for label, act, conf in SCENARIOS:
    ok, checks = authorize(act, GRANTED, confirmed=conf)
    failed = [k for k, v in checks.items() if not v]
    print(f'{label:<30}{str(ok):>6}  {failed if failed else "—"}')

assert authorize(Action('read', '/work/a.txt', sha('x'), L0), GRANTED)[0] is True
assert authorize(Action('read', '/home/u/.ssh/id_rsa', sha('x'), L0), GRANTED)[0] is False
assert authorize(Action('send', 'attacker@evil.test', sha('x'), L2), GRANTED, True)[0] is False
assert authorize(Action('send', 'boss@corp.test', sha('x'), L0), GRANTED, True)[0] is False
print('\n✅ 四个条件各自都能独立拦住一类动作——这是纵深防御在权限层的具体形态。')
print('   注意倒数两行：**范围**拦住了外部收件人，**信任门槛**拦住了 L0 上下文——')
print('   即使人已经点了确认。')

## 3 · 最小权限求解：从任务步骤反推

In [ ]:
TASK_STEPS = {
    '总结一个网页': [('fetch', '*.corp.test')],
    '整理工作目录里的笔记': [('read', '/work/**'), ('write', '/work/**')],
    '把笔记发给同事': [('read', '/work/**'), ('send', '*@corp.test')],
    '安装依赖并跑测试': [('fetch', '*.pypi.org'), ('read', '/work/**'), ('write', '/work/**')],
}

def minimal_caps(steps):
    """最小权限 = 每一步实际需要的能力的并集（且范围取最窄的那个）。"""
    needed = set()
    for op, target in steps:
        best = None
        for name, c in CAPS.items():
            if c.op != op:
                continue
            in_scope = (scope_match_safe(target.replace('**', 'x'), c.scope)
                        if target.startswith('/')
                        else any(glob_match(target.replace('*', 'x'), p) for p in c.scope))
            if in_scope:
                # 取范围最窄的（模式数最少 + 字符串最长 = 更具体）
                key = (len(c.scope), -sum(len(s) for s in c.scope))
                if best is None or key < best[0]:
                    best = (key, name)
        if best:
            needed.add(best[1])
    return sorted(needed)

ROLE_CODING_AGENT = sorted(CAPS)     # 「按角色授予」= 给它全部
print(f"{'任务':<24}{'最小权限':<46}{'规模':>6}")
for task, steps in TASK_STEPS.items():
    m = minimal_caps(steps)
    print(f'{task:<24}{str(m):<46}{len(m):>6}')
print(f'\n按角色授予（编码 agent）: {ROLE_CODING_AGENT}  规模 {len(ROLE_CODING_AGENT)}')

sizes = [len(minimal_caps(s)) for s in TASK_STEPS.values()]
print(f'最小权限的平均规模 {np.mean(sizes):.1f} vs 按角色 {len(ROLE_CODING_AGENT)}  '
      f'→ 过授 {len(ROLE_CODING_AGENT)/np.mean(sizes):.1f} 倍')
assert max(sizes) < len(ROLE_CODING_AGENT)
assert 'read_home' not in set().union(*[set(minimal_caps(s)) for s in TASK_STEPS.values()])
print('\n✅ 四个任务里没有一个需要 read_home——**而按角色授予会把它给出去**。')
print('   而这个权限正是「读 ~/.ssh/」的入口。')

In [ ]:
# 危险组合：这个权限集能不能构成外泄链路
def exfil_possible(granted):
    reads_private = any(CAPS[g].op == 'read' and CAPS[g].min_trust >= L2 for g in granted)
    # 更宽的定义：能读任何非工作目录的东西，或有 egress
    reads_any = any(CAPS[g].op == 'read' for g in granted)
    egress = any(CAPS[g].egress for g in granted)
    return {'reads': reads_any, 'reads_private': reads_private, 'egress': egress,
            'exfil': bool(reads_any and egress)}

print(f"{'权限集':<44}{'可读':>6}{'外通':>6}{'外泄链路':>10}")
for label, g in [('总结网页', minimal_caps(TASK_STEPS['总结一个网页'])),
                 ('整理笔记', minimal_caps(TASK_STEPS['整理工作目录里的笔记'])),
                 ('发笔记给同事', minimal_caps(TASK_STEPS['把笔记发给同事'])),
                 ('装依赖跑测试', minimal_caps(TASK_STEPS['安装依赖并跑测试'])),
                 ('按角色（全部）', ROLE_CODING_AGENT)]:
    e = exfil_possible(g)
    print(f'{label:<44}{str(e["reads"]):>6}{str(e["egress"]):>6}{str(e["exfil"]):>10}')

assert exfil_possible(minimal_caps(TASK_STEPS['整理工作目录里的笔记']))['exfil'] is False
assert exfil_possible(ROLE_CODING_AGENT)['exfil'] is True
print('\n✅ 「整理笔记」这个任务的最小权限集**不构成外泄链路**——')
print('   而按角色授予的权限集构成。')
print('   注意「发笔记给同事」与「装依赖跑测试」都构成链路，')
print('   这两个任务需要额外的收窄（范围限制 + 出站白名单，04 模块）。')

## 4 · 确认疲劳：确认频率如何摧毁确认的有效性

In [ ]:
def simulate_confirmation(n_requests, base_rate_malicious, prompts_per_hour,
                          careful0=0.95, decay=0.055, floor=0.10, seed=0):
    """人的认真程度随确认频率上升而衰减。
    careful = max(floor, careful0 * exp(-decay * prompts_per_hour))"""
    rng = np.random.default_rng(seed)
    careful = max(floor, careful0 * math.exp(-decay * prompts_per_hour))
    caught = missed = false_reject = 0
    for _ in range(n_requests):
        malicious = rng.random() < base_rate_malicious
        attentive = rng.random() < careful
        if malicious:
            (caught if attentive else globals())  # 占位，下面显式计数
            if attentive:
                caught += 1
            else:
                missed += 1
        else:
            # 认真看的时候偶尔也会误拒（很低）
            if attentive and rng.random() < 0.01:
                false_reject += 1
    n_mal = caught + missed
    return {'careful': careful, 'n_malicious': n_mal,
            'catch_rate': caught / n_mal if n_mal else float('nan'),
            'missed': missed, 'false_reject': false_reject}

print(f"{'每小时确认次数':>16}{'认真程度':>10}{'恶意请求':>10}{'拦住率':>10}{'漏放':>8}")
for pph in [1, 3, 10, 30, 60]:
    r = simulate_confirmation(4000, 0.01, pph, seed=1)
    print(f'{pph:>16}{r["careful"]:>10.1%}{r["n_malicious"]:>10}'
          f'{r["catch_rate"]:>10.1%}{r["missed"]:>8}')

r1 = simulate_confirmation(4000, 0.01, 1, seed=1)
r60 = simulate_confirmation(4000, 0.01, 60, seed=1)
assert r1['catch_rate'] > 0.85 and r60['catch_rate'] < 0.30
print(f'\n⚠️ 每小时 1 次确认时拦住 {r1["catch_rate"]:.0%}，'
      f'每小时 60 次时只拦住 {r60["catch_rate"]:.0%}。')
print('✅ **「给所有操作都加确认」不是更安全，而是更不安全**——')
print('   确认的有效性取决于它的信息密度，而不是频率。')
print('   （这与 C68-04 的告警疲劳完全同构。）')

In [ ]:
# 目标不是最大化确认次数，而是最大化「每次确认值得被看」的比例
def confirm_policy_compare(actions, policy):
    """policy: 'all' | 'irreversible' | 'irreversible_and_egress' | 'l0_only'"""
    prompted = []
    for act, cap_name, malicious in actions:
        cap = CAPS[cap_name]
        if policy == 'all':
            need = True
        elif policy == 'irreversible':
            need = not cap.reversible
        elif policy == 'irreversible_and_egress':
            need = (not cap.reversible) or cap.egress
        elif policy == 'l0_only':
            need = (act.ctx_trust == L0) and ((not cap.reversible) or cap.egress)
        else:
            raise ValueError(policy)
        if need:
            prompted.append(malicious)
    n = len(prompted)
    return {'n_prompts': n, 'base_rate': (sum(prompted) / n) if n else float('nan')}

rng = np.random.default_rng(3)
ACTIONS = []
for i in range(3000):
    cap_name = str(rng.choice(list(CAPS)))
    trust = int(rng.choice([L0, L1, L2], p=[0.35, 0.25, 0.40]))
    # 关键的建模选择：**恶意请求只出现在 L0 上下文里**——
    # 因为注入正是从不受信内容进来的（模块 01）。这不是简化，这是威胁模型。
    malicious = bool(trust == L0 and rng.random() < 0.03)
    ACTIONS.append((Action(CAPS[cap_name].op, 'x', sha(i), trust), cap_name, malicious))
print(f'构造 {len(ACTIONS)} 个动作，其中恶意 '
      f'{sum(1 for _, _, m in ACTIONS if m)} 个（全部来自 L0 上下文）')

print(f"{'确认策略':<28}{'确认次数':>10}{'其中恶意的比例(基率)':>22}")
res = {}
for pol in ['all', 'irreversible', 'irreversible_and_egress', 'l0_only']:
    r = confirm_policy_compare(ACTIONS, pol)
    res[pol] = r
    print(f'{pol:<28}{r["n_prompts"]:>10}{r["base_rate"]:>22.2%}')

assert res['all']['n_prompts'] > res['l0_only']['n_prompts'] * 3
assert res['l0_only']['base_rate'] > res['all']['base_rate']
print(f'\n✅ 从 all 收窄到 l0_only：确认次数降到 '
      f'{res["l0_only"]["n_prompts"]/res["all"]["n_prompts"]:.0%}，'
      f'而基率从 {res["all"]["base_rate"]:.2%} 升到 {res["l0_only"]["base_rate"]:.2%}。')
print('   **更少但更值得看的确认** —— 这才是提高确认有效性的方向。')

## 5 · TOCTOU：确认的内容与执行的内容不一致

In [ ]:
class ConfirmToken:
    def __init__(self, action_digest, by):
        self.action_digest = action_digest
        self.by = by
        self.used = False

class Executor:
    def __init__(self, strict=True):
        self.strict = strict
        self.log = []

    def execute(self, action, token):
        """strict=True 时校验「确认的 == 执行的」，并且令牌一次性。"""
        if self.strict:
            if token.used:
                return False, 'token_already_used'
            if token.action_digest != action.digest():
                return False, 'digest_mismatch(确认的与执行的不是同一个动作)'
            token.used = True
        self.log.append(action)
        return True, 'executed'

# 人确认的动作
approved = Action('send', 'boss@corp.test', sha('报告已完成'), L2)
tok = ConfirmToken(approved.digest(), by='user-42')

# 执行时参数被改了（参数重取 / 状态漂移）
tampered = Action('send', 'attacker@evil.test', sha('报告已完成'), L2)

for strict in [False, True]:
    ex = Executor(strict=strict)
    t = ConfirmToken(approved.digest(), 'user-42')
    ok1, why1 = ex.execute(tampered, t)
    print(f'strict={strict}: 执行被改过的动作 → {ok1} ({why1})')

ex_ok = Executor(strict=True)
t_ok = ConfirmToken(approved.digest(), 'user-42')
assert ex_ok.execute(tampered, t_ok)[0] is False
assert ex_ok.execute(approved, t_ok)[0] is True
# 令牌一次性
assert ex_ok.execute(approved, t_ok) == (False, 'token_already_used')
ex_bad = Executor(strict=False)
assert ex_bad.execute(tampered, ConfirmToken(approved.digest(), 'u'))[0] is True
print('\n✅ 三条防御同时生效：')
print('   ① frozen 的动作对象 —— 确认与执行之间不可能被修改')
print('   ② digest 校验     —— 「确认的 == 执行的」变成一个会失败的检查')
print('   ③ 令牌一次性       —— 重试路径上不会复用同一次确认')

In [ ]:
# 批量确认：范围必须显式限定，不能是开放式的
class BatchGrant:
    def __init__(self, op, scope, session, max_uses):
        self.op, self.scope, self.session = op, scope, session
        self.max_uses, self.uses = max_uses, 0

    def covers(self, action, session):
        if session != self.session or self.uses >= self.max_uses:
            return False
        if action.op != self.op:
            return False
        # 'ANY' 就是「不限目标」——这正是「不要再问我了」这个按钮的真实语义
        if 'ANY' in self.scope:
            return True
        return any(glob_match(action.target, p) for p in self.scope)

    def consume(self):
        self.uses += 1

# ❌ 开放式："send_email 不再询问" —— 范围由后续动作自己解释
OPEN = BatchGrant('send', ('ANY',), 'sess-1', max_uses=10 ** 9)
# ✓ 限定范围："本会话内，向 *@corp.test 发邮件不再询问，最多 20 次"
SCOPED = BatchGrant('send', ('*@corp.test',), 'sess-1', max_uses=20)

TESTS = [Action('send', 'boss@corp.test', sha('a'), L2),
         Action('send', 'attacker@evil.test', sha('b'), L2)]
print(f"{'动作':<34}{'开放式授权':>12}{'限定范围授权':>14}")
for a in TESTS:
    print(f'{a.op + " → " + a.target:<34}'
          f'{str(OPEN.covers(a, "sess-1")):>12}{str(SCOPED.covers(a, "sess-1")):>14}')
assert OPEN.covers(TESTS[1], 'sess-1') is True
assert SCOPED.covers(TESTS[1], 'sess-1') is False
assert SCOPED.covers(TESTS[0], 'sess-1') is True
assert SCOPED.covers(TESTS[0], 'sess-2') is False           # 跨会话失效
print('\n✅ 开放式授权把外部收件人也覆盖了——**攻击者只需让第一次操作看起来无害**。')
print('   限定范围的授权则只覆盖 *@corp.test，且绑定会话与次数上限。')

## 6 · 审计与 provenance 链

In [ ]:
class AuditLog:
    def __init__(self):
        self.entries = []
        self.seq = 0

    def pre(self, action, decision, checks, tool_manifest_sha, confirmation=None):
        """关键：**在执行之前**写入。否则执行中崩溃的动作没有任何记录。"""
        self.seq += 1
        e = {'seq': self.seq, 'phase': 'intent',
             'action': {'op': action.op, 'target': action.target,
                        'digest': action.digest()},
             'decision': decision, 'checks': checks,
             'ctx_trust': action.ctx_trust,
             'provenance': list(action.provenance),
             'confirmation': confirmation,
             'tool_manifest_sha': tool_manifest_sha}
        self.entries.append(e)
        return self.seq

    def post(self, seq, result):
        self.seq += 1
        self.entries.append({'seq': self.seq, 'phase': 'result',
                             'refs': seq, 'result': result})

    def gaps(self):
        seqs = [e['seq'] for e in self.entries]
        return [i for i in range(1, max(seqs, default=0) + 1) if i not in seqs]

    def trace(self, digest):
        """从一个动作反查它的完整来源链——事故复盘的核心操作。"""
        for e in self.entries:
            if e.get('action', {}).get('digest') == digest:
                return e['provenance']
        return None

PROV = (
    {'source': 'user', 'trust': L2, 'ref': 'msg-118'},
    {'source': 'web', 'trust': L0, 'ref': 'https://example.test/a'},
    {'source': 'subagent:summarizer', 'trust': L0, 'ref': 'sub-77'},
)
act = Action('send', 'boss@corp.test', sha('body'), L0, PROV)
audit = AuditLog()
ok, checks = authorize(act, GRANTED, confirmed=True)
s = audit.pre(act, 'allowed' if ok else 'denied', checks, tool_manifest_sha=sha(['t1', 't2']))
audit.post(s, 'blocked')

print('审计记录（意图阶段）:')
e = audit.entries[0]
for k in ['seq', 'phase', 'decision', 'ctx_trust', 'tool_manifest_sha']:
    print(f'  {k:<20} {e[k]}')
print('  provenance:')
for p in e['provenance']:
    print(f'    {LEVEL_NAME[p["trust"]]:<10} {p["source"]:<24} {p["ref"]}')

chain = audit.trace(act.digest())
assert chain is not None and len(chain) == 3
assert min(p['trust'] for p in chain) == L0
assert audit.gaps() == []
print(f'\n✅ 从动作反查来源链：最低信任等级 = {LEVEL_NAME[min(p["trust"] for p in chain)]}')
print('   → 这个动作是被一段 L0 网页内容（经子 agent 传递）触发的。')
print('   **provenance 是这份记录里唯一无法事后补上的字段**——')
print('   其余都能从配置与代码重建，而它如果当时没记就永远拿不回来。')

In [ ]:
# 日志缺口本身是一个告警
broken = AuditLog()
broken.entries = [{'seq': 1}, {'seq': 2}, {'seq': 5}]
broken.seq = 5
print('序号 [1,2,5] 的缺口:', broken.gaps())
assert broken.gaps() == [3, 4]
assert audit.gaps() == []
print('✅ 序号不连续 → 有记录丢失或被删 → 应当触发调查。')
print('   而这要求：**agent 不持有写审计日志的权限**，日志由包裹它的运行时写，只追加。')

## 7 · 变更影响面与季度锚点

In [ ]:
def surface_snapshot(granted):
    """当前权限集的攻击面快照。"""
    caps = [CAPS[g] for g in granted]
    reads = [g for g in granted if CAPS[g].op in ('read', 'fetch')]
    egress = [g for g in granted if CAPS[g].egress]
    irrev = [g for g in granted if not CAPS[g].reversible]
    return {'n_caps': len(granted),
            'danger_pairs': len(reads) * len(egress),
            'n_irreversible': len(irrev),
            'n_egress': len(egress),
            'exfil': bool(reads and egress)}

def change_impact(before, after):
    b, a = surface_snapshot(before), surface_snapshot(after)
    added = sorted(set(after) - set(before))
    severity = 'review'
    if not b['exfil'] and a['exfil']:
        severity = 'security-review'            # 从 0 变成非 0 —— 需要安全 review
    elif a['danger_pairs'] > b['danger_pairs']:
        severity = 'security-review'
    elif a['n_irreversible'] > b['n_irreversible']:
        severity = 'elevated'
    return {'added': added, 'before': b, 'after': a, 'severity': severity}

BASE_GRANT = ['read_workspace', 'write_workspace']
print(f"{'变更':<34}{'危险组合':>10}{'外泄链路':>10}{'需要什么 review':>18}")
for label, new in [('加 send_email', BASE_GRANT + ['send_email']),
                   ('加 http_get', BASE_GRANT + ['http_get']),
                   ('加 read_home', BASE_GRANT + ['read_home']),
                   ('加 http_get + read_home', BASE_GRANT + ['http_get', 'read_home'])]:
    ci = change_impact(BASE_GRANT, new)
    print(f'{label:<34}{ci["before"]["danger_pairs"]}→{ci["after"]["danger_pairs"]:<8}'
          f'{str(ci["before"]["exfil"])}→{str(ci["after"]["exfil"]):<6}{ci["severity"]:>18}')

ci_http = change_impact(BASE_GRANT, BASE_GRANT + ['http_get'])
ci_home = change_impact(BASE_GRANT, BASE_GRANT + ['read_home'])
assert ci_http['severity'] == 'security-review', '引入外泄链路必须走安全 review'
assert ci_home['severity'] in ('review', 'elevated')
print('\n✅ 「加 http_get」把外泄链路从不存在变成存在 → 自动要求安全 review。')
print('   这把「攻击面扩大」从一件靠人记得的事，变成了一次会失败的构建。')

In [ ]:
# 季度锚点：让「每次只加一点」的累积变得可见
QUARTERS = [
    ('2026-Q1', ['read_workspace', 'write_workspace']),
    ('2026-Q2', ['read_workspace', 'write_workspace', 'http_get']),
    ('2026-Q3', ['read_workspace', 'write_workspace', 'http_get', 'send_email']),
    ('2026-Q4', ['read_workspace', 'write_workspace', 'http_get', 'send_email', 'read_home']),
]
print(f"{'季度':<10}{'权限数':>8}{'危险组合':>10}{'不可逆':>8}{'出站':>8}{'外泄链路':>10}")
snaps = []
for q, g in QUARTERS:
    s = surface_snapshot(g)
    snaps.append((q, s))
    print(f'{q:<10}{s["n_caps"]:>8}{s["danger_pairs"]:>10}'
          f'{s["n_irreversible"]:>8}{s["n_egress"]:>8}{str(s["exfil"]):>10}')

first, last = snaps[0][1], snaps[-1][1]
print(f'\n一年内: 权限数 {first["n_caps"]}→{last["n_caps"]}，'
      f'危险组合 {first["danger_pairs"]}→{last["danger_pairs"]}')
# 每一步的增量
steps = [snaps[i+1][1]['danger_pairs'] - snaps[i][1]['danger_pairs'] for i in range(3)]
print(f'每季度的危险组合增量: {steps}  ← 每一步单独看都很小')
assert max(steps) <= 2 and last['danger_pairs'] >= 4 * max(first['danger_pairs'], 1)
print('\n✅ 每季度只增加 1–2 个危险组合，一年下来翻了几倍——')
print('   **这与 C68-03 的「渐进式退化」是同一个结构**：')
print('   每次单独看都合理，而累积效应没有任何人在看。')
print('   → 除了「相对上次」的检查，还需要一个季度锚点。')

## ✏️ 练习 1：范围检查的绕过测试集

实现 `scope_test_suite(matcher, patterns)`：对一组已知的绕过用例测试一个匹配器，
返回 `{'passed': [...], 'failed': [...], 'safe': bool}`。
用例格式 `(target, should_allow)`。

In [ ]:
SCOPE_CASES = [
    ('/work/a.txt', True),
    ('/work/sub/b.py', True),
    ('/work', True),
    ('/work/../etc/passwd', False),
    ('/work/./../../root/.ssh/id_rsa', False),
    ('/workspace_other/x', False),
    ('/home/u/.ssh/id_rsa', False),
    ('/work/../work/c.txt', True),          # 绕了一圈但仍在范围内
]

def scope_test_suite(matcher, patterns):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r_naive = scope_test_suite(scope_match_naive, WORKSPACE)
r_safe = scope_test_suite(scope_match_safe, WORKSPACE)
print(f'朴素匹配: safe={r_naive["safe"]}, 失败 {len(r_naive["failed"])} 例')
for t, exp, got in r_naive['failed']:
    print(f'   {t:<40} 期望={exp} 实际={got}')
print(f'\n安全匹配: safe={r_safe["safe"]}, 失败 {len(r_safe["failed"])} 例')
assert r_naive['safe'] is False and r_safe['safe'] is True
assert len(r_naive['failed']) >= 3
print('✅ 练习 1 通过：注意最后一个用例（/work/../work/c.txt）——')
print('   它绕了一圈但仍在范围内，**正确的匹配器必须允许它**。')
print('   一个「只要含 .. 就拒绝」的实现会在这里误拒。')

## ✏️ 练习 2：确认策略的效果对比

实现 `confirm_effectiveness(actions, policy, prompts_per_hour_scale=1/50)`：
先用 `confirm_policy_compare` 得到确认次数与基率，
再用 `simulate_confirmation` 估计拦住率（每小时确认次数 = 确认次数 × scale）。
返回 `{'n_prompts', 'base_rate', 'careful', 'catch_rate', 'expected_missed'}`。

In [ ]:
def confirm_effectiveness(actions, policy, prompts_per_hour_scale=1 / 50):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
print(f"{'策略':<28}{'确认次数':>10}{'基率':>9}{'认真度':>9}{'拦住率':>9}{'漏放':>7}")
eff = {}
for pol in ['all', 'irreversible', 'irreversible_and_egress', 'l0_only']:
    e = confirm_effectiveness(ACTIONS, pol)
    eff[pol] = e
    print(f'{pol:<28}{e["n_prompts"]:>10}{e["base_rate"]:>9.2%}'
          f'{e["careful"]:>9.1%}{e["catch_rate"]:>9.1%}{e["expected_missed"]:>7.1f}')
assert eff['l0_only']['careful'] > eff['all']['careful']
assert eff['l0_only']['expected_missed'] < eff['all']['expected_missed']
print(f'\n✅ 练习 2 通过：从 all 收窄到 l0_only，')
print(f'   确认次数减少 {1 - eff["l0_only"]["n_prompts"]/eff["all"]["n_prompts"]:.0%}，')
print(f'   而**漏放的恶意请求也减少了**（{eff["all"]["expected_missed"]:.1f} → '
      f'{eff["l0_only"]["expected_missed"]:.1f}）。')
print('   更少的确认换来更好的保护——这不矛盾，因为人的注意力是有限资源。')

## ✏️ 练习 3：TOCTOU 的检测

实现 `toctou_check(approved_action, executed_action, token)`：
返回 `(是否安全, 问题列表)`。检查三项：
① token 未被使用；② `token.action_digest == executed_action.digest()`；
③ `approved_action.digest() == executed_action.digest()`。

In [ ]:
def toctou_check(approved_action, executed_action, token):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
tok_a = ConfirmToken(approved.digest(), 'u')
ok_same, p_same = toctou_check(approved, approved, tok_a)
assert ok_same is True and p_same == []
ok_diff, p_diff = toctou_check(approved, tampered, ConfirmToken(approved.digest(), 'u'))
print('参数被改:', ok_diff, p_diff)
assert ok_diff is False and len(p_diff) >= 2
used = ConfirmToken(approved.digest(), 'u'); used.used = True
ok_used, p_used = toctou_check(approved, approved, used)
print('令牌已用过:', ok_used, p_used)
assert ok_used is False and any('used' in x for x in p_used)
print('\n✅ 练习 3 通过：三项检查覆盖了 TOCTOU 的三种主要形态——')
print('   参数重取 / 状态漂移（②③）与确认复用（①）。')

## ✏️ 练习 4：权限变更的自动分级

实现 `gate_permission_change(before, after, policy)`：
`policy` 是 `{严重度: 是否阻断}`。返回
`{'severity', 'blocked', 'added', 'delta_danger_pairs'}`。
用它验证「引入外泄链路的变更会被阻断」。

In [ ]:
def gate_permission_change(before, after, policy):
    # TODO：复用 change_impact
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
POLICY = {'security-review': True, 'elevated': False, 'review': False}
g1 = gate_permission_change(BASE_GRANT, BASE_GRANT + ['http_get'], POLICY)
g2 = gate_permission_change(BASE_GRANT, BASE_GRANT + ['read_home'], POLICY)
g3 = gate_permission_change(BASE_GRANT, BASE_GRANT, POLICY)
for label, g in [('加 http_get', g1), ('加 read_home', g2), ('无变更', g3)]:
    print(f'{label:<18} severity={g["severity"]:<18} blocked={g["blocked"]} '
          f'Δ危险组合={g["delta_danger_pairs"]:+d}')
assert g1['blocked'] is True and g1['delta_danger_pairs'] > 0
assert g2['blocked'] is False
assert g3['added'] == [] and g3['blocked'] is False
print('\n✅ 练习 4 通过：这个函数应当作为 CI 的一步——')
print('   引入外泄链路的权限变更会**阻断构建**，直到安全 review 通过。')
print('   这是 C68-04「确定性检查优先」在安全侧的直接应用：误报率天然为零。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def scope_test_suite(matcher, patterns):
    passed, failed = [], []
    for target, should_allow in SCOPE_CASES:
        got = matcher(target, patterns)
        if got == should_allow:
            passed.append(target)
        else:
            failed.append((target, should_allow, got))
    return {'passed': passed, 'failed': failed, 'safe': len(failed) == 0}

In [ ]:
# 练习 2 参考答案
def confirm_effectiveness(actions, policy, prompts_per_hour_scale=1 / 50):
    base = confirm_policy_compare(actions, policy)
    n = base['n_prompts']
    pph = n * prompts_per_hour_scale
    sim = simulate_confirmation(max(n, 1), base['base_rate'] if n else 0.0, pph, seed=7)
    return {'n_prompts': n, 'base_rate': base['base_rate'],
            'careful': sim['careful'], 'catch_rate': sim['catch_rate'],
            'expected_missed': float(sim['missed'])}

In [ ]:
# 练习 3 参考答案
def toctou_check(approved_action, executed_action, token):
    problems = []
    if token.used:
        problems.append('token_already_used')
    if token.action_digest != executed_action.digest():
        problems.append('token_digest != executed_digest')
    if approved_action.digest() != executed_action.digest():
        problems.append('approved_digest != executed_digest')
    return (len(problems) == 0, problems)

In [ ]:
# 练习 4 参考答案
def gate_permission_change(before, after, policy):
    ci = change_impact(before, after)
    sev = ci['severity']
    return {'severity': sev, 'blocked': bool(policy.get(sev, False)),
            'added': ci['added'],
            'delta_danger_pairs': ci['after']['danger_pairs'] - ci['before']['danger_pairs']}

---
## 🧪 真实工程胶囊：权限与沙箱的落地

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 能力声明：四个维度都要有（不要只有「有没有这个工具」）
# ══════════════════════════════════════════════════════════════════
CAPABILITIES = {
  "read_workspace": {"op": "read",  "scope": ["/work/**"],
                     "min_trust": "untrusted", "reversible": True},
  "send_email":     {"op": "send",  "scope": ["*@corp.test"],
                     "min_trust": "user", "reversible": False, "egress": True},
}
# 授权检查必须是四个条件的**合取**：
#   granted ∧ in_scope(target) ∧ ctx_trust >= min_trust ∧ (reversible ∨ confirmed)

# ══════════════════════════════════════════════════════════════════
# B. 路径范围检查：normpath + realpath，然后比较前缀 + 分隔符
# ══════════════════════════════════════════════════════════════════
def in_scope(path, roots):
    p = os.path.realpath(os.path.abspath(path))     # 解析 .. 与符号链接
    for r in roots:
        r = os.path.realpath(os.path.abspath(r))
        if p == r or p.startswith(r + os.sep):      # ← 必须加分隔符
            return True
    return False
# ❌ path.startswith("/work/")  会被 /work/../.ssh 与 /workspace_other 绕过
# 保留练习 1 的用例集作为回归测试。

# ══════════════════════════════════════════════════════════════════
# C. 沙箱：四条默认配置（都是「关掉」而不是「打开」）
# ══════════════════════════════════════════════════════════════════
# docker run \
#   --network=none \                       # ① 默认无网络，需要时走出站代理
#   --read-only \                          # ② 根文件系统只读
#   --tmpfs /work:rw,size=512m \           #    只有工作目录可写，任务间销毁重建
#   --cap-drop=ALL --security-opt=no-new-privileges \
#   --memory=2g --cpus=2 --pids-limit=256 \ # ④ 资源上限
#   <image>@sha256:...                      # 镜像 digest（C66-05）
#
# ③ **凭证不进沙箱**：agent 调 API 时经过一个持有凭证的代理，
#    沙箱内只有短期、窄 scope 的令牌（OAuth token exchange 的思路）。
#    这一条能把「凭证泄漏」这类最严重的后果直接消除。
#
# ⚠️ 记住沙箱的边界：它隔离的是**代码执行的副作用**，
#    而 agent 的主要能力来自被授权的 API 调用 —— 那些调用是"合法"的，沙箱不拦。

# ══════════════════════════════════════════════════════════════════
# D. 确认：动作对象冻结 + digest 校验 + 令牌一次性
# ══════════════════════════════════════════════════════════════════
@dataclass(frozen=True)                       # ← frozen 是关键
class Action:
    op: str; target: str; payload_sha: str; ctx_trust: int; provenance: tuple
    def digest(self): return sha256(...)

token = confirm_ui.request(action)            # UI 展示的是 action 的**关键字段**，
                                              # 不是「模型想调用 send_email」
assert not token.used
assert token.action_digest == action.digest() # ← 「确认的 == 执行的」
execute(action); token.used = True
#
# 确认策略：只对 (不可逆 ∨ 对外) ∧ ctx_trust==L0 的动作弹确认。
# 目标是**更少但更值得看的确认**——人的注意力是有限资源（练习 2）。
#
# 批量授权必须绑定范围 + 会话 + 次数上限：
#   「本会话内向 *@corp.test 发邮件不再询问，最多 20 次」
#   而不是「send_email 不再询问」。

# ══════════════════════════════════════════════════════════════════
# E. 审计：先记意图，再执行；provenance 是唯一补不回来的字段
# ══════════════════════════════════════════════════════════════════
seq = audit.pre(action, decision, checks, tool_manifest_sha, confirmation)
try:
    result = execute(action)
finally:
    audit.post(seq, result)          # 即使崩溃也有 intent 记录
#
# agent **不持有**写审计日志的权限；日志只追加，写在沙箱之外。
# 序号缺口本身是告警。

# ══════════════════════════════════════════════════════════════════
# F. CI 门禁：权限变更的自动分级（练习 4）
# ══════════════════════════════════════════════════════════════════
# 引入外泄链路（danger_pairs 从 0 变非 0）→ **阻断**，要求 security-review 标签
# 危险组合数增加                          → 阻断
# 不可逆操作数增加                        → elevated（提示 + 记录）
# 其余                                    → 普通 review
#
# 加上季度锚点：每季度记一次 surface_snapshot，与三个月前比较——
# 让「每次只加一点」的累积变得可见（与 C68-03 的渐进式退化同构）。
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 能力四维 | 操作 / 范围 / 信任门槛 / 可逆性，四条合取 → 四个独立收紧点 | 权限模型 |
| 范围这一维 | 最常被忽略也最有效；`startswith` 会被 `..` 与同前缀目录绕过 | 路径与域名检查 |
| 最小权限可计算 | 从任务步骤的并集反推；按角色授予通常过授几倍 | 授权设计 |
| 沙箱的边界 | 它隔离执行副作用，不隔离「被授权的 API 调用」 | 别把沙箱当万能 |
| 凭证不进沙箱 | 走持有凭证的代理，沙箱内只有短期窄 scope 令牌 | 消除最严重后果 |
| 确认疲劳 | 「给所有操作加确认」更不安全；目标是提高基率而非频率 | 确认策略 |
| TOCTOU | 冻结动作对象 + digest 校验 + 令牌一次性 | 确认实现 |
| provenance | 唯一无法事后补上的字段 | 审计设计 |
| 变更分级 + 季度锚点 | 让「攻击面扩大」变成一次会失败的构建 | CI 门禁 |

下一模块：**04 · 数据外泄与出站控制**——外泄的通道比你想的多得多
（渲染一张图片就够了），以及为什么出站白名单是「致命三要素」里最容易去掉的那一个。